# Семинар 3. Линейные модели: регрессия, классификация, регуляризация, SVM

В этом семинаре мы разберем:
- Линейную регрессию (аналитическое решение, градиентный спуск, SGD)
- Логистическую регрессию (бинарная и мультиклассовая классификация)
- Регуляризацию (Ridge, LASSO, их различия)
- Метод опорных векторов (SVM: линейный, ядровой, подбор параметров)

Для удобства линейные методы будем реализовывать в виде классов на Python наподобие соответствующих классов из sklearn.

In [ ]:
# Colab: install deps; locally use `uv run jupyter lab seminar.ipynb`
import sys
if "google.colab" in sys.modules:
    !pip install -q scikit-learn seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from matplotlib.colors import ListedColormap
from sklearn import datasets
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score

## 1. Линейная регрессия

Линейные методы предполагают, что между признаками объекта (features) и целевой переменной (target/label) существует линейная зависимость:
$$y = w_1 x_1 + w_2 x_2 + ... + w_k x_k + b,$$
где $y$ - целевая переменная, $x_i$ - признак объекта $x$, $w_i$ - вес $i$-го признака, $b$ - bias (смещение).

Часто предполагают, что объект $x$ содержит фиктивный признак равный 1 для представления свободного члена $b$:
$$y = \langle w, x \rangle,$$
где $\langle \cdot, \cdot \rangle$ - скалярное произведение векторов $w, x \in \mathbb{R}^n$.

В матричной форме для $n$ объектов:
$$ Y = Xw, $$
где $Y$ - столбец размера $n$, $X$ - матрица признаков размера $n \times k$, $w$ - вектор весов размера $k$.

**Лосс:**
$$
L(w) = \frac{1}{n}||Xw - Y||^2_2 = \frac{1}{n}\sum_{i=1}^{n}\left(\sum_{j=1}^{k} x_{ij}w_j - y_i\right)^2
$$

### 1.1 Аналитическое решение

Минимизация ошибки по методу наименьших квадратов дает решение:
$$ w = (X^TX)^{-1}X^TY $$

In [ ]:
class MyLinearRegression:
    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept

    def fit(self, X, y):
        n, k = X.shape
        X_train = X
        if self.fit_intercept:
            X_train = np.hstack((X, np.ones((n, 1))))
        self.w = np.linalg.inv(X_train.T @ X_train) @ X_train.T @ y
        return self

    def predict(self, X):
        n, k = X.shape
        if self.fit_intercept:
            X_train = np.hstack((X, np.ones((n, 1))))
        y_pred = X_train @ self.w
        return y_pred

    def get_weights(self):
        return self.w

#### Тестирование

Сгенерируем искусственные данные для теста моделей.

In [ ]:
def linear_expression(x):
    return 5 * x + 6

In [ ]:
objects_num = 50
X = np.linspace(-5, 5, objects_num)
y = linear_expression(X) + np.random.randn(objects_num) * 5

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5)

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(X, linear_expression(X), label='real', c='g')
plt.scatter(X_train, y_train, label='train', c='b')
plt.scatter(X_test, y_test, label='test', c='orange')
plt.title("Generated dataset")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

Обучим модель на трейне и предскажем результаты на тесте.

In [ ]:
regressor = MyLinearRegression()
regressor.fit(X_train[:, np.newaxis], y_train)
predictions = regressor.predict(X_test[:, np.newaxis])
w = regressor.get_weights()
w

In [ ]:
plt.figure(figsize=(20, 7))
ax = None
for i, types in enumerate([['train', 'test'], ['train'], ['test']]):
    ax = plt.subplot(1, 3, i + 1, sharey=ax)
    if 'train' in types:
        plt.scatter(X_train, y_train, label='train', c='b')
    if 'test' in types:
        plt.scatter(X_test, y_test, label='test', c='orange')
    plt.plot(X, linear_expression(X), label='real', c='g')
    plt.plot(X, regressor.predict(X[:, np.newaxis]), label='predicted', c='r')
    plt.ylabel('target')
    plt.xlabel('feature')
    plt.title(" ".join(types))
    plt.grid(alpha=0.2)
    plt.legend()
plt.show()

Сравним с реализацией из sklearn.

In [ ]:
sk_reg = LinearRegression().fit(X_train[:, np.newaxis], y_train)

plt.figure(figsize=(10, 7))
plt.plot(X, linear_expression(X), label='real', c='g')
plt.scatter(X_train, y_train, label='train')
plt.scatter(X_test, y_test, label='test')
plt.plot(X, regressor.predict(X[:, np.newaxis]), label='ours', c='r', linestyle=':')
plt.plot(X, sk_reg.predict(X[:, np.newaxis]), label='sklearn', c='cyan', linestyle=':')
plt.title("Different Prediction")
plt.ylabel('target')
plt.xlabel('feature')
plt.grid(alpha=0.2)
plt.legend()
plt.show()

#### Результаты

In [ ]:
train_predictions = regressor.predict(X_train[:, np.newaxis])
test_predictions = regressor.predict(X_test[:, np.newaxis])

print('Train MSE: ', mean_squared_error(y_train, train_predictions))
print('Test MSE: ', mean_squared_error(y_test, test_predictions))

### 1.2 Градиентный спуск

Обращение матрицы - очень долгая операция. Кроме того, обратная матрица $(X^TX)^{-1}$ может вообще не существовать (вырожденная матрица). Поэтому часто используют итеративные методы оптимизации - градиентный спуск.

Градиент MSE:
$$\frac{\partial{L}}{\partial{w}} = \frac{2}{n}X^T(Xw - Y)$$

Шаг обновления:
$$w_{t+1} = w_t - \rho \cdot \frac{\partial{L}}{\partial{w}}$$

где $\rho$ - learning rate (скорость обучения).

Визуализация градиентного спуска: "шарик" скатывается по поверхности лосса к минимуму.

In [ ]:
# Визуализация градиентного спуска на контурном графике
def loss_surface(w0, w1):
    return (w0 - 5) ** 2 + 3 * (w1 - 6) ** 2

w0_grid = np.linspace(-5, 15, 200)
w1_grid = np.linspace(-4, 16, 200)
W0, W1 = np.meshgrid(w0_grid, w1_grid)
Z = loss_surface(W0, W1)

# Запустим GD
lr = 0.05
w = np.array([-3.0, 14.0])
path = [w.copy()]
for _ in range(30):
    grad = np.array([2 * (w[0] - 5), 6 * (w[1] - 6)])
    w = w - lr * grad
    path.append(w.copy())
path = np.array(path)

fig, ax = plt.subplots(figsize=(10, 7))
ax.contour(W0, W1, Z, levels=30, cmap='viridis', alpha=0.6)
ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=4, linewidth=1.5, label='GD trajectory')
ax.plot(5, 6, 'k*', markersize=15, label='minimum')
ax.set_xlabel('$w_0$')
ax.set_ylabel('$w_1$')
ax.set_title('Gradient Descent on Loss Surface')
ax.legend()
ax.grid(alpha=0.2)
plt.show()

Реализуем класс линейной регрессии с градиентным спуском.

In [ ]:
class MyGradientLinearRegression(MyLinearRegression):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.w = None

    def fit(self, X, y, lr=0.01, max_iter=100):
        n, k = X.shape
        if self.w is None:
            self.w = np.random.randn(k + 1 if self.fit_intercept else k)
        X_train = np.hstack((X, np.ones((n, 1)))) if self.fit_intercept else X
        self.losses = []
        for iter_num in range(max_iter):
            y_pred = self.predict(X)
            self.losses.append(mean_squared_error(y_pred, y))
            grad = self._calc_gradient(X_train, y, y_pred)
            assert grad.shape == self.w.shape, f"gradient shape {grad.shape} is not equal weight shape {self.w.shape}"
            self.w -= lr * grad
        return self

    def _calc_gradient(self, X, y, y_pred):
        grad = 2 * (y_pred - y)[:, np.newaxis] * X
        grad = grad.mean(axis=0)
        return grad

    def get_losses(self):
        return self.losses

#### Тестирование

In [ ]:
regressor = MyGradientLinearRegression(fit_intercept=True)
l = regressor.fit(X_train[:, np.newaxis], y_train, max_iter=100).get_losses()
predictions = regressor.predict(X_test[:, np.newaxis])
w = regressor.get_weights()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(X, linear_expression(X), label='real', c='g')
plt.scatter(X_train, y_train, label='train')
plt.scatter(X_test, y_test, label='test')
plt.plot(X, regressor.predict(X[:, np.newaxis]), label='predicted', c='r')
plt.grid(alpha=0.2)
plt.legend()
plt.show()

Построим также график лосса во время обучения.

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(l)
plt.title('Gradient descent learning')
plt.ylabel('loss')
plt.xlabel('iteration')
plt.ylim(bottom=0)
plt.grid(alpha=0.2)
plt.show()

### 1.3 Стохастический градиентный спуск (SGD)

Добавим к градиентному спуску сэмплирование мини-батча, по которому будем считать градиент.

In [ ]:
class MySGDLinearRegression(MyGradientLinearRegression):
    def __init__(self, n_sample=10, **kwargs):
        super().__init__(**kwargs)
        self.w = None
        self.n_sample = n_sample

    def _calc_gradient(self, X, y, y_pred):
        inds = np.random.choice(np.arange(X.shape[0]), size=self.n_sample, replace=False)
        grad = 2 * (y_pred[inds] - y[inds])[:, np.newaxis] * X[inds]
        grad = grad.mean(axis=0)
        return grad

#### Тестирование

In [ ]:
regressor = MySGDLinearRegression(fit_intercept=True)
l = regressor.fit(X_train[:, np.newaxis], y_train, max_iter=100).get_losses()
predictions = regressor.predict(X_test[:, np.newaxis])
w = regressor.get_weights()

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(X, linear_expression(X), label='real', c='g')
plt.scatter(X_train, y_train, label='train')
plt.scatter(X_test, y_test, label='test')
plt.plot(X, regressor.predict(X[:, np.newaxis]), label='predicted', c='r')
plt.grid(alpha=0.2)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(l)
plt.title('SGD learning')
plt.ylabel('loss')
plt.xlabel('iteration')
plt.grid(alpha=0.2)
plt.show()

Протестируем, меняя размер мини-батча.

In [ ]:
n_samples = [1, 2, 4]

plt.figure(figsize=(10, 7))
for ns in n_samples:
    l = MySGDLinearRegression(fit_intercept=True, n_sample=ns).fit(
        X_train[:, np.newaxis], y_train, lr=5e-3, max_iter=150,
    ).get_losses()
    plt.plot(l, alpha=0.5, label=f'{ns} mini-batch size')

plt.title('SGD: effect of mini-batch size')
plt.ylabel('loss')
plt.xlabel('iteration')
plt.legend()
plt.ylim((0, 150))
plt.grid(alpha=0.2)
plt.show()

Как видно по графикам, размер подвыборки влияет на стабильность сходимости: чем меньше батч, тем больше шума в градиенте.

Критерии остановки SGD:
- Количество итераций достигло максимума
- Learning rate не меняется/уменьшается
- Loss не меняется
- Норма градиента $\approx 0$

## 2. Логистическая регрессия

Задача аналогична линейной регрессии, только мы переводим выходное значение в вероятность принадлежности к классу.

### 2.1 От линейной регрессии к классификации

Линейная регрессия дает $y \in (-\infty; +\infty)$. Нам нужно $p \in [0; 1]$.

Цепочка преобразований:

| Величина | Область значений |
|---|---|
| Вероятность $p$ | $[0; 1]$ |
| Шансы (odds) $\frac{p}{1-p}$ | $[0; +\infty)$ |
| Логарифм шансов (log-odds) $\log\frac{p}{1-p}$ | $(-\infty; +\infty)$ |

Обратное преобразование из log-odds в вероятность дает сигмоидную функцию:
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

In [ ]:
# Сигмоидная функция
x_sig = np.linspace(-6, 6, 200)
y_sig = 1 / (1 + np.exp(-x_sig))

plt.figure(figsize=(10, 5))
plt.plot(x_sig, y_sig, 'b-', linewidth=2)
plt.axhline(y=0, color='k', linewidth=0.5)
plt.axhline(y=1, color='k', linewidth=0.5, linestyle='--')
plt.axhline(y=0.5, color='gray', linewidth=0.5, linestyle='--')
plt.axvline(x=0, color='gray', linewidth=0.5, linestyle='--')
plt.xlabel('$z = \\langle w, x \\rangle$')
plt.ylabel('$\\sigma(z)$')
plt.title('Sigmoid function')
plt.grid(alpha=0.2)
plt.show()

Задача формулируется так:

**Предсказания:**
$$y_{pred}(x, w) = \sigma(\langle x, w \rangle) = \frac{1}{1 + e^{-\langle x, w \rangle}}$$

**Лосс (binary cross-entropy / log-loss):**
$$L(w) = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(y_{pred_i}) + (1 - y_i)\log(1 - y_{pred_i})\right]$$

In [ ]:
# Log-loss: штраф за неправильную уверенность
p = np.linspace(0.01, 0.99, 200)

plt.figure(figsize=(10, 5))
plt.plot(p, -np.log(p), 'b-', linewidth=2, label='$-\\log(p)$: loss when $y=1$')
plt.plot(p, -np.log(1 - p), 'r-', linewidth=2, label='$-\\log(1-p)$: loss when $y=0$')
plt.xlabel('predicted probability $p$')
plt.ylabel('loss')
plt.title('Log-loss components')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

**Градиент:**
$$\frac{\partial{L}}{\partial{w}} = \frac{1}{n}X^T(\sigma(Xw) - Y)$$

### 2.2 Реализация

In [ ]:
def logit(x, w):
    return np.dot(x, w)

def sigmoid(h):
    return 1. / (1 + np.exp(-h))

class MyLogisticRegression:
    def __init__(self):
        self.w = None

    def fit(self, X, y, max_iter=100, lr=0.1):
        n, k = X.shape
        if self.w is None:
            self.w = np.random.randn(k + 1)
        X_train = np.concatenate((np.ones((n, 1)), X), axis=1)
        losses = []
        for iter_num in range(max_iter):
            z = sigmoid(logit(X_train, self.w))
            grad = np.dot(X_train.T, (z - y)) / len(y)
            self.w -= grad * lr
            losses.append(self.__loss(y, z))
        return losses

    def predict_proba(self, X):
        n, k = X.shape
        X_ = np.concatenate((np.ones((n, 1)), X), axis=1)
        return sigmoid(logit(X_, self.w))

    def predict(self, X, threshold=0.5):
        return self.predict_proba(X) >= threshold

    def get_weights(self):
        return self.w

    def __loss(self, y, p):
        p = np.clip(p, 1e-10, 1 - 1e-10)
        return np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

In [ ]:
X, y = make_blobs(n_samples=1000, centers=[[-2, 0.5], [2, -0.5]], cluster_std=1, random_state=42)

colors = ("red", "green")
colored_y = np.zeros(y.size, dtype=str)
for i, cl in enumerate([0, 1]):
    colored_y[y == cl] = str(colors[i])

plt.figure(figsize=(15, 10))
plt.scatter(X[:, 0], X[:, 1], c=colored_y)
plt.show()

In [ ]:
clf = MyLogisticRegression()
clf.fit(X, y, max_iter=1000)
w = clf.get_weights()
w

In [ ]:
plt.figure(figsize=(15, 8))
eps = 0.1
xx, yy = np.meshgrid(
    np.linspace(np.min(X[:, 0]) - eps, np.max(X[:, 0]) + eps, 500),
    np.linspace(np.min(X[:, 1]) - eps, np.max(X[:, 1]) + eps, 500),
)
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA'])
plt.pcolormesh(xx, yy, Z, cmap=cmap_light)
plt.scatter(X[:, 0], X[:, 1], c=colored_y)
plt.title('Decision boundary (MyLogisticRegression)')
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))
Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.pcolormesh(xx, yy, Z, cmap=plt.get_cmap('viridis'))
colors = ("magenta", "green")
colored_y2 = np.zeros(y.size, dtype=str)
for i, cl in enumerate([0, 1]):
    colored_y2[y == cl] = str(colors[i])
plt.scatter(X[:, 0], X[:, 1], c=colored_y2)
plt.colorbar()
plt.title('Predicted probabilities')
plt.show()

### 2.3 Мультиклассовая классификация (Iris)

Протестируем логистическую регрессию на реальных данных с тремя классами.

In [ ]:
iris = load_iris()
data = pd.DataFrame(
    data=np.hstack([iris.data, iris.target[:, np.newaxis]]),
    columns=iris.feature_names + ['target'],
)
names = data.columns
data.head()

In [ ]:
sns.pairplot(data, hue='target')
plt.show()

In [ ]:
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    data[names[:-1]], data[names[-1]], random_state=42,
)

In [ ]:
cls = make_pipeline(StandardScaler(), LogisticRegression(C=2))
cls.fit(X_train_iris.to_numpy(), y_train_iris)

preds_train = cls.predict(X_train_iris)
print('Train:', accuracy_score(preds_train, y_train_iris), f1_score(preds_train, y_train_iris, average='macro'))

preds_test = cls.predict(X_test_iris)
print('Test: ', accuracy_score(preds_test, y_test_iris), f1_score(preds_test, y_test_iris, average='macro'))

In [ ]:
# Decision boundary по двум первым признакам
plt.figure(figsize=(15, 8))
eps = 0.1
xx, yy = np.meshgrid(
    np.linspace(np.min(X_train_iris[names[0]]) - eps, np.max(X_train_iris[names[0]]) + eps, 500),
    np.linspace(np.min(X_train_iris[names[1]]) - eps, np.max(X_train_iris[names[1]]) + eps, 500),
)

cls.fit(X_train_iris[names[:2]], y_train_iris)
Z = cls.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

cmap_light = ListedColormap(['#AAAAFF', '#FFAAAA', '#AAFFAA'])
plt.pcolormesh(xx, yy, Z, cmap=cmap_light)
plt.scatter(X_train_iris[names[0]], X_train_iris[names[1]], c=y_train_iris, cmap='brg')
plt.xlabel(names[0])
plt.ylabel(names[1])
plt.title('Logistic Regression: decision boundary (Iris, 2 features)')
plt.show()

## 3. Регуляризация

Зачастую модель обучается на зашумленных данных. Веса подбираются для минимизации ошибки на трейне, но могут быть слишком большими и подстраиваться под шум (переобучение). Регуляризация штрафует модель за большие веса, заставляя искать более простые решения.

Еще одна проблема - **мультиколлинеарность**: когда признаки сильно скоррелированы, матрица $X^TX$ близка к вырожденной, и веса становятся неустойчивыми.

Два основных типа регуляризации:
- **L2 (Ridge):** штраф на сумму квадратов весов $\|w\|_2^2$
- **L1 (LASSO):** штраф на сумму модулей весов $\|w\|_1$

In [ ]:
# Данные для экспериментов с регуляризацией
objects_num = 50
X = np.linspace(-5, 5, objects_num)
y = linear_expression(X) + np.random.randn(objects_num) * 5
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5)

### 3.1 Ridge регрессия (L2-регуляризация)

В Ridge мы штрафуем модель на сумму квадратов весов:
$$L(w) = \frac{1}{n}\|Xw - Y\|^2_2 + \alpha\|w\|^2_2$$

Аналитическое решение:
$$w = (X^TX + \alpha I)^{-1}X^TY$$

Интересная особенность Ridge: из-за формулировки лосса, утроение данных с alpha эквивалентно alpha/3 на исходных данных.

In [ ]:
# Ridge на утроенных данных с alpha=1
reg = Ridge(alpha=1).fit(np.hstack((X, X, X))[:, np.newaxis], np.hstack((y, y, y)))
print('Tripled data, alpha=1:', np.append(reg.coef_, reg.intercept_))

# Ridge на исходных данных с alpha=1/3
reg = Ridge(alpha=1/3).fit(X[:, np.newaxis], y)
print('Original data, alpha=1/3:', np.append(reg.coef_, reg.intercept_))

#### 3.1.1 Аналитическое решение

In [ ]:
class MyRidgeRegression(MyLinearRegression):
    def __init__(self, alpha=1.0, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha

    def fit(self, X, y):
        n, m = X.shape
        X_train = X
        if self.fit_intercept:
            X_train = np.hstack((X, np.ones((n, 1))))
        lambdaI = self.alpha * np.eye(X_train.shape[1])
        if self.fit_intercept:
            lambdaI[-1, -1] = 0
        self.w = np.linalg.inv(X_train.T @ X_train + lambdaI) @ X_train.T @ y
        return self

    def get_weights(self):
        return self.w

In [ ]:
alpha = 1.0
regressor = MyRidgeRegression(alpha=alpha).fit(X_train[:, np.newaxis], y_train)
sklearn_reg = Ridge(alpha).fit(X_train[:, np.newaxis], y_train)
assert np.allclose(regressor.get_weights(), np.append(sklearn_reg.coef_, sklearn_reg.intercept_))
regressor.get_weights()

In [ ]:
plt.figure(figsize=(20, 7))
ax = None
for i, types in enumerate([['train', 'test'], ['train'], ['test']]):
    ax = plt.subplot(1, 3, i + 1, sharey=ax)
    if 'train' in types:
        plt.scatter(X_train, y_train, label='train', c='b')
    if 'test' in types:
        plt.scatter(X_test, y_test, label='test', c='orange')
    plt.plot(X, linear_expression(X), label='real', c='g')
    plt.plot(X, regressor.predict(X[:, np.newaxis]), label='predicted', c='r')
    plt.ylabel('target')
    plt.xlabel('feature')
    plt.title(" ".join(types))
    plt.grid(alpha=0.2)
    plt.legend()
plt.show()

#### 3.1.2 SGD

Градиент Ridge:
$$\frac{\partial{L}}{\partial{w}} = \frac{2}{n}X^T(Xw - Y) + 2\alpha w$$

In [ ]:
class MySGDRidge(MySGDLinearRegression):
    def __init__(self, alpha=1.0, **kwargs):
        super().__init__(**kwargs)
        self.w = None
        self.alpha = alpha

    def _calc_gradient(self, X, y, y_pred):
        inds = np.random.choice(np.arange(X.shape[0]), size=self.n_sample, replace=False)
        lambdaI = self.alpha * np.eye(self.w.shape[0])
        if self.fit_intercept:
            lambdaI[-1, -1] = 0
        grad = 2 * (X[inds].T @ X[inds] / self.n_sample + lambdaI) @ self.w
        grad -= 2 * X[inds].T @ y[inds] / self.n_sample
        return grad

In [ ]:
regressor = MySGDRidge(alpha=1, n_sample=20).fit(X[:, np.newaxis], y, max_iter=1000, lr=0.01)
l = regressor.get_losses()
regressor.get_weights()

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(X, linear_expression(X), label='real', c='g')
plt.scatter(X_train, y_train, label='train')
plt.scatter(X_test, y_test, label='test')
plt.plot(X, regressor.predict(X[:, np.newaxis]), label='predicted', c='r')
plt.grid(alpha=0.2)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(l)
plt.title('Ridge learning with SGD')
plt.ylabel('loss')
plt.xlabel('iteration')
plt.grid(alpha=0.2)
plt.show()

### 3.2 LASSO регрессия (L1-регуляризация)

В LASSO мы штрафуем модель на сумму модулей весов (L1-норма):
$$L(w) = \frac{1}{n}\|Xw - Y\|^2_2 + \alpha\|w\|_1$$

В отличие от Ridge, аналитическое решение LASSO в общем случае не находится в замкнутом виде. Используем SGD.

В LASSO наблюдается желаемое поведение: утроение данных не влияет на коэффициенты так, как в Ridge.

In [ ]:
reg = Lasso(alpha=1).fit(np.hstack((X, X, X))[:, np.newaxis], np.hstack((y, y, y)))
print('Tripled data:', np.append(reg.coef_, reg.intercept_))

reg = Lasso(alpha=1).fit(X[:, np.newaxis], y)
print('Original data:', np.append(reg.coef_, reg.intercept_))

#### 3.2.1 SGD

Градиент L1-нормы - это знак весов (subgradient, т.к. $|w|$ недифференцируема в нуле):
$$\frac{\partial\|w\|_1}{\partial w_j} = \text{sign}(w_j)$$

Используем сглаженную версию `soft_sign` для численной стабильности.

In [ ]:
def soft_sign(x, eps=1e-7):
    if abs(x) > eps:
        return np.sign(x)
    return x / eps

np_soft_sign = np.vectorize(soft_sign)

class MySGDLasso(MySGDLinearRegression):
    def __init__(self, alpha=1.0, **kwargs):
        super().__init__(**kwargs)
        self.w = None
        self.alpha = alpha

    def _calc_gradient(self, X, y, y_pred):
        inds = np.random.choice(np.arange(X.shape[0]), size=self.n_sample, replace=False)
        signw = np_soft_sign(self.w)
        if self.fit_intercept:
            signw[-1] = 0
        grad = X[inds].T @ (y_pred[inds] - y[inds])[:, np.newaxis] / self.n_sample
        grad += self.alpha * signw[:, np.newaxis]
        return grad.flatten()

In [ ]:
regressor = MySGDLasso(alpha=1, n_sample=4).fit(X[:, np.newaxis], y, max_iter=1000, lr=0.01)
l = regressor.get_losses()
print('Our weights:', regressor.get_weights())

sklearn_reg = Lasso().fit(X[:, np.newaxis], y)
print('Sklearn weights:', np.append(sklearn_reg.coef_, sklearn_reg.intercept_))

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(X, linear_expression(X), label='real', c='g')
plt.scatter(X_train, y_train, label='train')
plt.scatter(X_test, y_test, label='test')
plt.plot(X, regressor.predict(X[:, np.newaxis]), label='predicted', c='r')
plt.grid(alpha=0.2)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(l)
plt.title('LASSO learning with SGD')
plt.ylabel('loss')
plt.xlabel('iteration')
plt.grid(alpha=0.2)
plt.show()

### 3.3 Сравнение Ridge и LASSO

Геометрическая интерпретация: L1-ограничение (ромб) чаще "касается" осей, что приводит к обнулению весов (разреженность). L2-ограничение (круг) равномерно сжимает веса.

In [ ]:
# Геометрическая интерпретация L1 vs L2
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Контуры эллиптического лосса
w1 = np.linspace(-2, 2, 300)
w2 = np.linspace(-2, 2, 300)
W1, W2 = np.meshgrid(w1, w2)
Z = 2 * (W1 - 0.8) ** 2 + (W2 - 1.5) ** 2

for ax, title in zip(axes, ['L1 (LASSO)', 'L2 (Ridge)']):
    ax.contour(W1, W2, Z, levels=15, cmap='viridis', alpha=0.5)
    ax.axhline(y=0, color='k', linewidth=0.3)
    ax.axvline(x=0, color='k', linewidth=0.3)
    ax.set_xlabel('$w_1$')
    ax.set_ylabel('$w_2$')
    ax.set_title(title)
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')

# L1 ball (diamond)
r = 1.0
diamond = plt.Polygon([[r, 0], [0, r], [-r, 0], [0, -r]], fill=True, alpha=0.3, color='blue')
axes[0].add_patch(diamond)
axes[0].plot(0, r, 'ro', markersize=8)  # solution on axis = sparse

# L2 ball (circle)
circle = plt.Circle((0, 0), r, fill=True, alpha=0.3, color='blue')
axes[1].add_patch(circle)
axes[1].plot(0.35, 0.94, 'ro', markersize=8)  # solution not on axis

plt.tight_layout()
plt.show()

Основные различия между L1 и L2 регуляризациями:

- LASSO **сложнее считать** из-за недифференцируемых углов ограничения
- LASSO **обнуляет веса** (feature selection), Ridge только сжимает их
- Ridge имеет **аналитическое решение**, LASSO - нет
- На практике часто используют **Elastic Net** - комбинацию L1 и L2

## 4. Метод опорных векторов (SVM)

SVM (Support Vector Machine) - метод, который ищет гиперплоскость с максимальным отступом (margin) от ближайших точек каждого класса. Эти ближайшие точки называются опорными векторами (support vectors).

Задача оптимизации:
$$\min_{w, b} \frac{1}{2}\|w\|^2 \quad \text{subject to} \quad y_i(\langle w, x_i \rangle + b) \geq 1$$

На практике данные редко линейно разделимы, поэтому используют soft margin с параметром $C$:
$$\min_{w, b} \frac{1}{2}\|w\|^2 + C\sum_i \xi_i$$

Параметр $C$ управляет компромиссом между шириной отступа и количеством ошибок.

### 4.1 Линейный SVM

In [ ]:
X_svm, y_svm = make_blobs(n_samples=200, centers=2, cluster_std=1.2, random_state=42)

In [ ]:
def plot_svm_decision_boundary(clf, X, y, ax=None, title=None):
    """Plot decision boundary, margins, and support vectors for SVM."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))
    eps = 0.5
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - eps, X[:, 0].max() + eps, 300),
        np.linspace(X[:, 1].min() - eps, X[:, 1].max() + eps, 300),
    )
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-1e10, 0, 1e10], colors=['#FFAAAA', '#AAAAFF'], alpha=0.4)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors=['red', 'black', 'blue'], linestyles=['--', '-', '--'])
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolors='k', s=30)
    if hasattr(clf, 'support_vectors_'):
        ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
                   s=150, facecolors='none', edgecolors='k', linewidths=2, label='support vectors')
    if title:
        ax.set_title(title)
    ax.grid(alpha=0.2)
    return ax

In [ ]:
svm_linear = SVC(kernel='linear', C=1.0)
svm_linear.fit(X_svm, y_svm)

plot_svm_decision_boundary(svm_linear, X_svm, y_svm, title='Linear SVM (C=1.0)')
plt.legend()
plt.show()

**Влияние параметра C:** маленький C - широкий margin (больше ошибок допускается), большой C - узкий margin (модель строже).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, C in zip(axes, [0.1, 1.0, 100.0]):
    svm = SVC(kernel='linear', C=C).fit(X_svm, y_svm)
    plot_svm_decision_boundary(svm, X_svm, y_svm, ax=ax, title=f'C = {C}')
plt.tight_layout()
plt.show()

### 4.2 Нелинейный SVM (ядровые методы)

Для нелинейно разделимых данных SVM использует kernel trick - отображение данных в пространство более высокой размерности, где они становятся линейно разделимыми. При этом само отображение явно не вычисляется - вместо этого используется ядерная функция $K(x_i, x_j)$.

Основные ядра:
- **RBF (Radial Basis Function):** $K(x_i, x_j) = \exp(-\gamma\|x_i - x_j\|^2)$ - универсальное ядро
- **Polynomial:** $K(x_i, x_j) = (\gamma \langle x_i, x_j \rangle + r)^d$ - полиномиальные границы

In [ ]:
# Нелинейно разделимые данные
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)

In [ ]:
# Сравнение ядер на make_moons
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
kernels = [('linear', {}), ('rbf', {'gamma': 'scale'}), ('poly', {'degree': 3})]

for ax, (kernel, params) in zip(axes, kernels):
    svm = SVC(kernel=kernel, C=1.0, **params).fit(X_moons, y_moons)
    plot_svm_decision_boundary(svm, X_moons, y_moons, ax=ax, title=f'kernel={kernel}')

plt.suptitle('SVM kernels on make_moons', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# То же на make_circles
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (kernel, params) in zip(axes, kernels):
    svm = SVC(kernel=kernel, C=1.0, **params).fit(X_circles, y_circles)
    plot_svm_decision_boundary(svm, X_circles, y_circles, ax=ax, title=f'kernel={kernel}')

plt.suptitle('SVM kernels on make_circles', fontsize=14)
plt.tight_layout()
plt.show()

### 4.3 Подбор параметров

Для RBF ядра ключевые параметры:
- **C** - штраф за ошибки (маленький C = простая модель, большой C = сложная)
- **gamma** - "радиус влияния" опорных векторов (маленький gamma = гладкая граница, большой gamma = сложная/переобученная)

In [ ]:
# Grid: C vs gamma для RBF SVM на make_moons
C_values = [0.1, 1, 10]
gamma_values = [0.1, 1, 10]

fig, axes = plt.subplots(len(C_values), len(gamma_values), figsize=(18, 16))

for i, C in enumerate(C_values):
    for j, gamma in enumerate(gamma_values):
        ax = axes[i][j]
        svm = SVC(kernel='rbf', C=C, gamma=gamma).fit(X_moons, y_moons)
        plot_svm_decision_boundary(svm, X_moons, y_moons, ax=ax, title=f'C={C}, gamma={gamma}')
        if i == 0:
            ax.set_title(f'gamma={gamma}\nC={C}', fontsize=12)
        if j == 0:
            ax.set_ylabel(f'C={C}', fontsize=12)

plt.suptitle('RBF SVM: effect of C and gamma', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Наблюдения:
- Маленький gamma + маленький C: underfitting (слишком простая модель)
- Большой gamma + большой C: overfitting (граница повторяет шум)
- Оптимум где-то посередине

На практике для подбора параметров используют `GridSearchCV` из sklearn.

## 5. Итоги

| Метод | Задача | Регуляризация | Аналитическое решение | Особенности |
|---|---|---|---|---|
| Linear Regression | Регрессия | Нет | Да | Базовый метод |
| Ridge | Регрессия | L2 | Да | Сжимает веса |
| LASSO | Регрессия | L1 | Нет | Обнуляет веса (feature selection) |
| Logistic Regression | Классификация | L1/L2 | Нет (итеративно) | Предсказывает вероятности |
| SVM (linear) | Классификация | C | Нет (QP) | Максимальный отступ |
| SVM (kernel) | Классификация | C, gamma | Нет (QP) | Нелинейные границы через kernel trick |

### Когда что использовать

- **Линейная регрессия** - когда зависимость линейная, данных немного
- **Ridge** - когда много скоррелированных признаков, нужна стабильность
- **LASSO** - когда нужен автоматический отбор признаков
- **Logistic Regression** - базовый классификатор, интерпретируемый, быстрый
- **SVM** - когда нужна нелинейная граница, данных не слишком много (SVM плохо масштабируется на >100k объектов)